In [50]:
# Standard Library
import re
import urllib.error
import urllib.request
from pathlib import Path
from matplotlib.patches import Polygon

# Data
import numpy as np
import pandas as pd

# Visualisierung
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Jupyter
import ipywidgets as widgets
from IPython.display import display

# Hockey rink
from hockey_rink import NHLRink

# Setup
rink = NHLRink()

pd.options.display.max_columns = None

# To Dos

Rink rotation automatisch machen
von wo aus? -> Shot Event bestimmt darstellung des Rinks
Rotation hinterfragen. Macht 90/270 grad rotation sinn? -> evtl sinnvoller normal belassen.

Was genau darstellen? Ganzer rink oder nur offense / defense zonen? evtl. ozone / dzone
https://github.com/the-bucketless/hockey_rink/blob/master/examples/drawing.ipynb

herausfinden weiso gewisse einträge keine direction haben.¨

polygon darstellung wenn goalieonice = false

In [51]:
# Daten laden -> Pfad Anpassen
data_dir = Path("/Users/enrue/Developer/playground/hockey-dataset/data/")
df = pd.read_csv(data_dir / "canada_v_hcd_shotData16.csv")

df["strength_type_state"] = df["TeamStrengthType"].ffill()

In [52]:
#Config

#Goalie ID aktuell noch manuell
# evtl. spieler welcher über zeit am nächsten beim tor ist automatisch als torwart deklarieren?
attacking_goalie_id = 304441.0
defending_goalie_id = 888052815.0

# Frame Range
start_frame = 0
shot_frame = 9

# Attacker Colors
attacker_color = "dodgerblue"

# Defender Colors
defender_color = "seagreen"
initial_polygon_color = "seagreen"
final_polygon_color = "seagreen"

# Event Colors
puck_control_color = "dodgerblue"
pass_color = "crimson"

In [53]:
# Jede Event-Zeile bekommt einen fortlaufenden Frame-Index
structure_df = df.reset_index(drop=True).copy()

# Vorerst alle verfügbaren Rows verwenden
start_frame = 0
shot_frame = len(structure_df) - 1

pre_shot_df = structure_df.iloc[
    start_frame:shot_frame + 1
].copy()

In [54]:
# Koordinaten von Metern in Feet umrechnen
M_TO_FT = 3.280839895

def coordinates_to_feet(series):
    coords = (
        series
        .astype("string")
        .str.split(",", n=1, expand=True)
        # damit leere Koordinaten nicht zum absturz führen
        .reindex(columns=[0, 1])
    )

    x = pd.to_numeric(coords[0], errors="coerce") * M_TO_FT
    y = pd.to_numeric(coords[1], errors="coerce") * M_TO_FT
    return x, y

In [55]:
# Event-Koordinaten in Feet umrechnen
df["EventStartX_ft"], df["EventStartY_ft"] = coordinates_to_feet(df["EventStartCoordinate"])
df["EventEndX_ft"], df["EventEndY_ft"] = coordinates_to_feet(df["EventEndCoordinate"])

In [56]:
# Spielerkoordinaten in Feet umrechnen
for i in range(1, 13):
    coord_col = f"StartPlayerCoordinates{i}"
    if coord_col not in df.columns:
        continue
        
    x, y = coordinates_to_feet(df[coord_col])

    df[f"StartPlayerX{i}_ft"] = x
    df[f"StartPlayerY{i}_ft"] = y

In [57]:
# Richtungskoordinaten in X und Y aufteilen
def split_xy(series):
    coords = (
        series
        .astype("string")
        .str.split(",", n=1, expand=True)
        .reindex(columns=[0, 1])
    )

    x = pd.to_numeric(coords[0], errors="coerce")
    y = pd.to_numeric(coords[1], errors="coerce")

    return x, y

In [58]:
# Spieler-Richtungen aufspalten
for i in range(1, 13):
    dir_col = f"StartPlayerDirection{i}"

    if dir_col not in df.columns:
        continue

    dx, dy = split_xy(df[dir_col])

    df[f"StartPlayerDirX{i}"] = dx
    df[f"StartPlayerDirY{i}"] = dy

In [59]:
#Frame als Index (jede Row ein Frame)
structure_df = df.reset_index(drop=True).copy()

# Pre-Shot-Fenster auswählen
pre_shot_df = structure_df.iloc[start_frame:shot_frame + 1].copy()

In [60]:
# Prüft ob PuckControl und Pass denselben weg haben (Versuch) 
def is_redundant_puck_control(
    previous_row,
    current_row,
    coordinate_tolerance_ft=1.0,
    time_tolerance_ms=150
):

    if not (
        previous_row["EventType"] == "Pass"
        and current_row["EventType"] == "PuckControl"
    ):
        return False

    coordinate_columns = [
        "EventStartX_ft",
        "EventStartY_ft",
        "EventEndX_ft",
        "EventEndY_ft"
    ]

    if (
        previous_row[coordinate_columns].isna().any()
        or current_row[coordinate_columns].isna().any()
    ):
        return False

    time_difference = abs(
        current_row["Timestamp"]
        - previous_row["Timestamp"]
    )

    if time_difference > time_tolerance_ms:
        return False

    start_distance = np.hypot(
        current_row["EventStartX_ft"]
        - previous_row["EventStartX_ft"],
        current_row["EventStartY_ft"]
        - previous_row["EventStartY_ft"]
    )

    end_distance = np.hypot(
        current_row["EventEndX_ft"]
        - previous_row["EventEndX_ft"],
        current_row["EventEndY_ft"]
        - previous_row["EventEndY_ft"]
    )

    return (
        start_distance <= coordinate_tolerance_ft
        and end_distance <= coordinate_tolerance_ft
    )

In [61]:
# Spielernummer aus Namen wie "#11 Danny Nelson" extrahieren
def extract_player_number(name):
    if pd.isna(name):
        return ""

    match = re.match(r"#(\d+)", str(name).strip())
    return match.group(1) if match else str(name)

In [62]:
# Position der Spieler extrahieren
def collect_team_positions(pre_shot_df, target_team):
    team_positions = []

    for frame_idx, row in pre_shot_df.iterrows():

        for i in range(1, 13):
            team = row.get(f"StartPlayerTeam{i}")
            player_id = row.get(f"StartPlayerId{i}")
            player_name = row.get(f"StartPlayerName{i}")

            x = row.get(f"StartPlayerX{i}_ft")
            y = row.get(f"StartPlayerY{i}_ft")

            dir_x = row.get(f"StartPlayerDirX{i}")
            dir_y = row.get(f"StartPlayerDirY{i}")

            # Nur Spieler des gewünschten Teams
            # mit vorhandener Position übernehmen
            if (
                team != target_team
                or pd.isna(player_id)
                or pd.isna(player_name)
                or pd.isna(x)
                or pd.isna(y)
            ):
                continue

            team_positions.append({
                "frame": frame_idx,
                "player_id": player_id,
                "player_name": player_name,
                "player_number": extract_player_number(player_name),
                "x": x,
                "y": y,
                "dir_x": dir_x,
                "dir_y": dir_y
            })

    return pd.DataFrame(team_positions)

In [63]:
# Hinweis! Direction_length dient nur als hilfe, könnte auch 10 sein
# Evtl. ganz weglassen da nicht alle Events direction der Spieler haben

def get_plot_angle(x, y, dir_x, dir_y, ax, direction_length=3):
    if pd.isna(dir_x) or pd.isna(dir_y):
        return None

    # Ausgangspunkt
    plot_x, plot_y = rink.convert_xy([x], [y], ax=ax)
    plot_x = plot_x[0]
    plot_y = plot_y[0]

    # Hilfspunkt in Blickrichtung
    x2 = x + dir_x * direction_length
    y2 = y + dir_y * direction_length

    plot_x2, plot_y2 = rink.convert_xy([x2], [y2], ax=ax)
    plot_x2 = plot_x2[0]
    plot_y2 = plot_y2[0]

    angle_rad = np.arctan2(
        plot_y2 - plot_y,
        plot_x2 - plot_x
    )

    return angle_rad

In [64]:
#
def draw_direction_triangle(
    ax,
    plot_x,
    plot_y,
    angle_rad,
    facecolor,
    edgecolor="black",
    length=2.0,
    width=1.6,
    alpha=1.0,
    zorder=6
):
    if angle_rad is None:
        return

    ux = np.cos(angle_rad)
    uy = np.sin(angle_rad)

    px = -uy
    py = ux

    tip = (
        plot_x + length * ux,
        plot_y + length * uy
    )

    base_center = (
        plot_x - 0.8 * length * ux,
        plot_y - 0.8 * length * uy
    )

    left = (
        base_center[0] + 0.5 * width * px,
        base_center[1] + 0.5 * width * py
    )

    right = (
        base_center[0] - 0.5 * width * px,
        base_center[1] - 0.5 * width * py
    )

    triangle = Polygon(
        [tip, left, right],
        closed=True,
        facecolor=facecolor,
        edgecolor=edgecolor,
        linewidth=1.0,
        alpha=alpha,
        zorder=zorder
    )

    ax.add_patch(triangle)

In [65]:
# Bewegungsverlauf der Spieler zeichnen
def draw_team_trails(
    ax,
    team_df,
    color,
    show_final_number=True
):
    if team_df.empty:
        return

    # Jeden Spieler einzeln durchgehen
    for player_id, player_df in team_df.groupby("player_id"):

        player_df = (
            player_df
            .sort_values("frame")
            .reset_index(drop=True)
            .copy()
        )

        n_positions = len(player_df)

        # Spielerpositionen passend zum gezeichneten Rink transformieren
        plot_x, plot_y = rink.convert_xy(
            player_df["x"].to_numpy(),
            player_df["y"].to_numpy(),
            ax=ax
        )

        plot_x = np.asarray(plot_x)
        plot_y = np.asarray(plot_y)

        # Opacity von früh -> spät
        alphas = np.linspace(
            0.4,
            1.0,
            n_positions
        )

        # Bewegungslinien
        for i in range(n_positions - 1):
            ax.plot(
                [plot_x[i], plot_x[i + 1]],
                [plot_y[i], plot_y[i + 1]],
                color=color,
                linewidth=2.0,
                alpha=alphas[i + 1],
                zorder=3
            )

        # Frühere Positionen als Direction-Dreiecke
        for i in range(n_positions - 1):

            current_row = player_df.iloc[i]

            angle_rad = get_plot_angle(
                x=current_row["x"],
                y=current_row["y"],
                dir_x=current_row["dir_x"],
                dir_y=current_row["dir_y"],
                ax=ax
            )

            if angle_rad is not None:
                draw_direction_triangle(
                    ax=ax,
                    plot_x=plot_x[i],
                    plot_y=plot_y[i],
                    angle_rad=angle_rad,
                    facecolor=color,
                    alpha=alphas[i],
                    zorder=4
                )

            # Keine Direction vorhanden -> Punkt als Fallback
            else:
                ax.scatter(
                    plot_x[i],
                    plot_y[i],
                    s=50,
                    color=color,
                    edgecolors="black",
                    linewidth=0.7,
                    alpha=alphas[i],
                    zorder=4
                )

        # Finale Position
        final_row = player_df.iloc[-1]

        final_plot_x = plot_x[-1]
        final_plot_y = plot_y[-1]

        player_number = final_row["player_number"]

        # Finale Position als Punkt
        ax.scatter(
            final_plot_x,
            final_plot_y,
            s=150,
            color=color,
            edgecolors="black",
            linewidth=1.2,
            alpha=1.0,
            zorder=50
        )

        # Spielernummer im finalen Punkt
        if show_final_number:
            ax.text(
                final_plot_x,
                final_plot_y,
                str(player_number),
                ha="center",
                va="center",
                fontsize=8,
                fontweight="bold",
                color="white",
                zorder=100,
                clip_on=False
            )

In [66]:
def get_powerplay_teams(strength_type):
    if strength_type == "HomePowerplay":
        return "Home", "Away"

    if strength_type == "AwayPowerplay":
        return "Away", "Home"

    return None, None

In [67]:
# Verteidigungs-Polygon -> Verbindet die vier Spielerpositionen eines Frames zu einem Polygon
def draw_structure_polygon(
    ax,
    frame_df,
    color,
    alpha=0.15,
    linewidth=2,
    linestyle="-"
):

    if len(frame_df) != 4:
        return

    plot_x, plot_y = rink.convert_xy(
        frame_df["x"].to_numpy(),
        frame_df["y"].to_numpy(),
        ax=ax
    )

    center_x = np.mean(plot_x)
    center_y = np.mean(plot_y)

    # Punkte um den Mittelpunkt sortieren
    angles = np.arctan2(
        plot_y - center_y,
        plot_x - center_x
    )

    order = np.argsort(angles)

    polygon_coordinates = [
        (plot_x[i], plot_y[i])
        for i in order
    ]

    polygon = Polygon(
        polygon_coordinates,
        closed=True,
        facecolor=color,
        edgecolor=color,
        alpha=alpha,
        linewidth=linewidth,
        linestyle=linestyle,
        zorder=2
    )

    ax.add_patch(polygon)

In [68]:
def plot_pre_shot_summary(
    pre_shot_df,
    attacking_goalie_id=None,
    defending_goalie_id=None,
    show_passes=True,
    show_puck_control=True,
    show_attacker_trails=True,
    show_defender_trails=True,
    show_initial_polygon=True,
    show_final_polygon=True
):
    fig, ax = plt.subplots(figsize=(11, 8))

    # Rink automatisch ausrichten (under construction)
    display_range, rotation, event_position = get_rink_view(
        pre_shot_df
    )

    rink.draw(
        display_range=display_range,
        rotation=rotation,
        ax=ax
    )

    # Teams bestimmen
    first_row = pre_shot_df.iloc[0]

    attacking_team, defending_team = get_powerplay_teams(
        first_row["strength_type_state"]
    )

    # Events zeichnen
    for i in range(len(pre_shot_df)):

        row = pre_shot_df.iloc[i]
        event_df = pre_shot_df.iloc[[i]]

        # Wenn Passes angezeigt werden, haben sie Vorrang vor einem praktisch identischen PuckControl.
        # Bei einer reinen PuckControl-Ansicht bleiben hingegen alle PuckControls sichtbar.
        if (
            show_passes
            and row["EventType"] == "PuckControl"
            and i > 0
        ):
            previous_row = pre_shot_df.iloc[i - 1]

            if is_redundant_puck_control(
                previous_row,
                row
            ):
                continue

        # PuckControl
        if (
            show_puck_control
            and row["EventType"] == "PuckControl"
        ):
            rink.wavy_arrow(
                data=event_df,
                x="EventStartX_ft",
                y="EventStartY_ft",
                x2="EventEndX_ft",
                y2="EventEndY_ft",
                head_width=2,
                length_includes_head=True,
                color=puck_control_color,
                alpha=0.75,
                ax=ax
            )

        # Pass
        elif (
            show_passes
            and row["EventType"] == "Pass"
        ):
            rink.arrow(
                data=event_df,
                x="EventStartX_ft",
                y="EventStartY_ft",
                x2="EventEndX_ft",
                y2="EventEndY_ft",
                head_width=0,
                length_includes_head=True,
                color=pass_color,
                alpha=0.75,
                ax=ax
            )

        # Shot
        elif row["EventType"] == "Shot":
            rink.scatter(
                data=event_df,
                x="EventStartX_ft",
                y="EventStartY_ft",
                s=260,
                marker="*",
                color="gold",
                edgecolors="black",
                zorder=7,
                ax=ax
            )

    # Spielerpositionen sammeln
    attacking_positions_df = collect_team_positions(
        pre_shot_df=pre_shot_df,
        target_team=attacking_team
    )

    defending_positions_df = collect_team_positions(
        pre_shot_df=pre_shot_df,
        target_team=defending_team
    )

    # Goalies herausfiltern -> Id muss definiert sein
    if attacking_goalie_id is not None:
        attacking_positions_df = attacking_positions_df[
            attacking_positions_df["player_id"] != attacking_goalie_id
        ].copy()

    if defending_goalie_id is not None:
        defending_positions_df = defending_positions_df[
            defending_positions_df["player_id"] != defending_goalie_id
        ].copy()

    # Trails zeichnen
    if show_attacker_trails:
        draw_team_trails(
            ax=ax,
            team_df=attacking_positions_df,
            color=attacker_color
        )

    if show_defender_trails:
        draw_team_trails(
            ax=ax,
            team_df=defending_positions_df,
            color=defender_color
        )

    # Start- und Endpolygon
    first_frame = defending_positions_df["frame"].min()
    last_frame = defending_positions_df["frame"].max()

    initial_defenders = defending_positions_df[
        defending_positions_df["frame"] == first_frame
    ]

    final_defenders = defending_positions_df[
        defending_positions_df["frame"] == last_frame
    ]

    if show_initial_polygon:
        draw_structure_polygon(
            ax=ax,
            frame_df=initial_defenders,
            color=initial_polygon_color,
            alpha=0.1,
            linewidth=1.5,
            linestyle="--"
        )

    if show_final_polygon:
        draw_structure_polygon(
            ax=ax,
            frame_df=final_defenders,
            color=final_polygon_color,
            alpha=0.25,
            linewidth=2.5,
            linestyle="-"
        )

    # Titel + Legende (Spielernamen noch ergänzen)


    shot_rows = pre_shot_df[
        pre_shot_df["EventType"] == "Shot"
    ]

    shot_row = (
        shot_rows.iloc[-1]
        if not shot_rows.empty
        else pre_shot_df.iloc[-1]
    )

    shooter = shot_row.get(
        "EventPrimaryPlayerName",
        "Unknown shooter"
    )
    
    legend_elements = [
        Line2D(
            [0], [0],
            color=attacker_color,
            linewidth=2.5,
            label="Attacker movement / puck actions"
        ),
        Line2D(
            [0], [0],
            color=defender_color,
            linewidth=2.5,
            label="Defender movement"
        ),
        Line2D(
            [0], [0],
            marker="*",
            linestyle="None",
            markerfacecolor="gold",
            markeredgecolor="black",
            markersize=14,
            label="Shot"
        )
    ]

    ax.legend(
        handles=legend_elements,
        loc="upper left",
        bbox_to_anchor=(1.02, 1),
        title="Legend"
    )

    ax.set_title(
        f"Pre-shot sequence | "
        f"Frames {pre_shot_df.index.min()}–{pre_shot_df.index.max()} | "
        f"Shooter: {shooter} | "
        f"{event_position}"
    )

    plt.show()

In [69]:
# Angriffzone immer Oben
def get_rink_view(pre_shot_df):

    # Schuss als Referenz suchen
    shot_rows = pre_shot_df[
        pre_shot_df["EventType"] == "Shot"
    ]

    if not shot_rows.empty:
        reference_row = shot_rows.iloc[-1]
    else:
        reference_row = pre_shot_df.iloc[-1]

    event_position = reference_row.get("EventPosition")

    # HomeTeamZone
    if event_position == "HomeTeamZone":
        display_range = "defence"
        rotation = 270

    # AwayTeamZone
    elif event_position == "AwayTeamZone":
        display_range = "offence"
        rotation = 90


    # Neutral Zone (evtl unnötig -> Abhängig von Input Daten)
    elif event_position == "NeutralZone":
        display_range = "full"
        rotation = 90

    # Fallback
    else:
        display_range = "full"
        rotation = 90

    return display_range, rotation, event_position

In [70]:
# nur für debug
display_range, rotation, event_position = get_rink_view(
    pre_shot_df
)

print("EventPosition:", event_position)
print("Display range:", display_range)
print("Rotation:", rotation)

EventPosition: HomeTeamZone
Display range: defence
Rotation: 270


In [71]:
# Toggle Buttons für Visualisierung
show_passes_widget = widgets.Checkbox(
    value=True,
    description="Passes"
)

show_puck_control_widget = widgets.Checkbox(
    value=True,
    description="PuckControl"
)

show_attacker_trails_widget = widgets.Checkbox(
    value=True,
    description="Attacker trails"
)

show_defender_trails_widget = widgets.Checkbox(
    value=True,
    description="Defender trails"
)

show_initial_polygon_widget = widgets.Checkbox(
    value=True,
    description="Start polygon"
)

show_final_polygon_widget = widgets.Checkbox(
    value=True,
    description="Shot polygon"
)

In [72]:
def render_pre_shot_view(
    show_passes,
    show_puck_control,
    show_attacker_trails,
    show_defender_trails,
    show_initial_polygon,
    show_final_polygon
):

    plot_pre_shot_summary(
        pre_shot_df=pre_shot_df,
        attacking_goalie_id=attacking_goalie_id,
        defending_goalie_id=defending_goalie_id,
        show_passes=show_passes,
        show_puck_control=show_puck_control,
        show_attacker_trails=show_attacker_trails,
        show_defender_trails=show_defender_trails,
        show_initial_polygon=show_initial_polygon,
        show_final_polygon=show_final_polygon
    )

In [73]:
# Was im Visual angezeigt wird
pre_shot_df

,Unnamed: 0,Period,MatchClock,Timestamp,EventType,TeamStrengthType,TeamStrength,HomeGoalieOnIce,AwayGoalieOnIce,EventPosition,EventStartCoordinate,EventEndCoordinate,EventPrimaryPlayerId,EventPrimaryPlayerName,EventPrimaryTeam,EventSecondaryPlayerId,EventSecondaryPlayerName,EventSecondaryTeam,FaceoffSpot,ShotResult,ShotSpeed,ShotScreeningPlayerIds,ShotXG,GoalAssist1Id,GoalAssist2Id,PassResult,PassSpeed,BluelineCrossingResult,BluelineCrossingDirection,BluelineCrossingAttackerIds,BluelineCrossingDefenderIds,DumpResult,DumpType,DumpAlongTheBoards,ClearType,ClearAlongTheBoards,PuckControlState,PuckControlStartZone,PuckControlEndZone,PuckControlContestingPlayerIds,StoppageOfPlayReason,PenaltyReason,ParticipatingPlayerIds,StartPlayerId1,StartPlayerId2,StartPlayerId3,StartPlayerId4,StartPlayerId5,StartPlayerId6,StartPlayerId7,StartPlayerId8,StartPlayerId9,StartPlayerId10,StartPlayerId11,StartPlayerId12,StartPlayerName1,StartPlayerName2,StartPlayerName3,StartPlayerName4,StartPlayerName5,StartPlayerName6,StartPlayerName7,StartPlayerName8,StartPlayerName9,StartPlayerName10,StartPlayerName11,StartPlayerName12,StartPlayerTeam1,StartPlayerTeam2,StartPlayerTeam3,StartPlayerTeam4,StartPlayerTeam5,StartPlayerTeam6,StartPlayerTeam7,StartPlayerTeam8,StartPlayerTeam9,StartPlayerTeam10,StartPlayerTeam11,StartPlayerTeam12,StartPlayerCoordinates1,StartPlayerCoordinates2,StartPlayerCoordinates3,StartPlayerCoordinates4,StartPlayerCoordinates5,StartPlayerCoordinates6,StartPlayerCoordinates7,StartPlayerCoordinates8,StartPlayerCoordinates9,StartPlayerCoordinates10,StartPlayerCoordinates11,StartPlayerCoordinates12,StartPlayerVelocity1,StartPlayerVelocity2,StartPlayerVelocity3,StartPlayerVelocity4,StartPlayerVelocity5,StartPlayerVelocity6,StartPlayerVelocity7,StartPlayerVelocity8,StartPlayerVelocity9,StartPlayerVelocity10,StartPlayerVelocity11,StartPlayerVelocity12,StartPlayerDirection1,StartPlayerDirection2,StartPlayerDirection3,StartPlayerDirection4,StartPlayerDirection5,StartPlayerDirection6,StartPlayerDirection7,StartPlayerDirection8,StartPlayerDirection9,StartPlayerDirection10,StartPlayerDirection11,StartPlayerDirection12,strength_type_state,EventStartX_ft,EventStartY_ft,EventEndX_ft,EventEndY_ft,StartPlayerX1_ft,StartPlayerY1_ft,StartPlayerX2_ft,StartPlayerY2_ft,StartPlayerX3_ft,StartPlayerY3_ft,StartPlayerX4_ft,StartPlayerY4_ft,StartPlayerX5_ft,StartPlayerY5_ft,StartPlayerX6_ft,StartPlayerY6_ft,StartPlayerX7_ft,StartPlayerY7_ft,StartPlayerX8_ft,StartPlayerY8_ft,StartPlayerX9_ft,StartPlayerY9_ft,StartPlayerX10_ft,StartPlayerY10_ft,StartPlayerX11_ft,StartPlayerY11_ft,StartPlayerX12_ft,StartPlayerY12_ft,StartPlayerDirX1,StartPlayerDirY1,StartPlayerDirX2,StartPlayerDirY2,StartPlayerDirX3,StartPlayerDirY3,StartPlayerDirX4,StartPlayerDirY4,StartPlayerDirX5,StartPlayerDirY5,StartPlayerDirX6,StartPlayerDirY6,StartPlayerDirX7,StartPlayerDirY7,StartPlayerDirX8,StartPlayerDirY8,StartPlayerDirX9,StartPlayerDirY9,StartPlayerDirX10,StartPlayerDirY10,StartPlayerDirX11,StartPlayerDirY11,StartPlayerDirX12,StartPlayerDirY12
0,3571,3,516,1.703800e+12,Pass,AwayPowerplay,4v5,True,True,HomeTeamZone,"-18.6,-13","-9.5,-6.4",337830.0,#44 Matej Stransky,Away,308599.0,#46 Dominik Egli,Away,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Successful,14.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"337,830,308,599",304441.0,143575.0,337830.0,340163.0,140525.0,308599.0,888052818.0,333800.0,343628.0,888052815.0,888052819.0,NaN,#91 Gilles Senn,#70 Enzo Corvi,#44 Matej Stransky,#42 Joakim Nordström,#65 Marc Wieser,#46 Dominik Egli,#3 Dillon Heatherington,#86 Josh Jooris,#19 Corban Knight,#30 Aaron Dell,#24 Ty Smith,NaN,Away,Away,Away,Away,Away,Away,Home,Home,Home,Home,Home,NaN,"25.7,0.2","-13.7,2.6","-19.1,-13","-23.7,2.4","-22,-3.1","-7.6,-4.9","-23.4,-0.5","-16.6,-3.7","-18.7,-2.4","-25.8,-0.9","-20.5,-7.9",NaN,0.420,4.002,0.533,1.025,2.290,3.258,0.919,0.179,1.348,0.327,1.246,NaN,"0.209,0.978","0.464,0.886","0.896,0.444","-0.448,-0.894","-0.997,-0.074","-0.162,-0.987","-0.762,0.648"

In [74]:
controls = widgets.VBox([
    widgets.HBox([
        show_passes_widget,
        show_puck_control_widget
    ]),
    widgets.HBox([
        show_attacker_trails_widget,
        show_defender_trails_widget
    ]),
    widgets.HBox([
        show_initial_polygon_widget,
        show_final_polygon_widget
    ])
])

output = widgets.interactive_output(
    render_pre_shot_view,
    {
        "show_passes": show_passes_widget,
        "show_puck_control": show_puck_control_widget,
        "show_attacker_trails": show_attacker_trails_widget,
        "show_defender_trails": show_defender_trails_widget,
        "show_initial_polygon": show_initial_polygon_widget,
        "show_final_polygon": show_final_polygon_widget
    }
)

display(controls, output)

Output()